In [1]:
import numpy as np
import tensorflow as tf
from tensorflow import keras


In [2]:
# Function to load the data from file
def load_data(filename):
    sentences = []
    labels = []
    with open(filename, 'r', encoding='utf-8') as file:
        sentence = []
        label = []
        for line in file:
            if line == '\n':
                if sentence:
                    sentences.append(sentence)
                    labels.append(label)
                    sentence = []
                    label = []
                continue
            token, entity, _ = line.strip().split('\t')
            sentence.append(token)
            label.append(entity)
    return sentences, labels

In [3]:
# Load the training and test data
train_sentences, train_labels = load_data('en_ewt_nn_train_answers_only.conll')
test_sentences, test_labels = load_data('en_ewt_nn_answers_test.conll')

In [4]:
# Tokenization and Preprocessing
tokenizer = keras.preprocessing.text.Tokenizer(lower=True, oov_token='OOV')
tokenizer.fit_on_texts(train_sentences)
vocab_size = len(tokenizer.word_index) + 1

train_sequences = tokenizer.texts_to_sequences(train_sentences)
test_sequences = tokenizer.texts_to_sequences(test_sentences)

max_len = max(len(seq) for seq in train_sequences + test_sequences)

X_train = keras.preprocessing.sequence.pad_sequences(train_sequences, maxlen=max_len, padding='post')
X_test = keras.preprocessing.sequence.pad_sequences(test_sequences, maxlen=max_len, padding='post')

In [5]:
# Convert labels to one-hot encoding
label_encoder = keras.preprocessing.text.Tokenizer(lower=False)
label_encoder.fit_on_texts(train_labels)

num_labels = len(label_encoder.word_index) + 1

y_train = keras.preprocessing.sequence.pad_sequences(label_encoder.texts_to_sequences(train_labels), maxlen=max_len, padding='post')
y_train = keras.utils.to_categorical(y_train, num_labels)

y_test = keras.preprocessing.sequence.pad_sequences(label_encoder.texts_to_sequences(test_labels), maxlen=max_len, padding='post')
y_test = keras.utils.to_categorical(y_test, num_labels)

In [6]:
# Split training data into training and validation sets
validation_split = 0.2
split_index = int(len(X_train) * (1 - validation_split))

X_train, X_val = X_train[:split_index], X_train[split_index:]
y_train, y_val = y_train[:split_index], y_train[split_index:]

In [7]:
# Step 2: Word Embedding
embedding_dim = 100

model = keras.Sequential([
    keras.layers.Embedding(vocab_size, embedding_dim, input_length=max_len),
    keras.layers.Bidirectional(keras.layers.LSTM(units=64, return_sequences=True)),
    keras.layers.Dense(num_labels, activation='softmax')
])


In [8]:
# Step 3: Model Compilation
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])


In [9]:
# Step 4: Model Training
epochs = 10
batch_size = 32

model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=epochs, batch_size=batch_size)


Epoch 1/10
66/66 [==============================] - 15s 116ms/step - loss: 0.4874 - accuracy: 0.9048 - val_loss: 0.0558 - val_accuracy: 0.9910
Epoch 2/10
66/66 [==============================] - 6s 87ms/step - loss: 0.0503 - accuracy: 0.9915 - val_loss: 0.0298 - val_accuracy: 0.9966
Epoch 3/10
66/66 [==============================] - 6s 89ms/step - loss: 0.0387 - accuracy: 0.9939 - val_loss: 0.0242 - val_accuracy: 0.9970
Epoch 4/10
66/66 [==============================] - 6s 89ms/step - loss: 0.0327 - accuracy: 0.9948 - val_loss: 0.0204 - val_accuracy: 0.9973
Epoch 5/10
66/66 [==============================] - 6s 89ms/step - loss: 0.0284 - accuracy: 0.9949 - val_loss: 0.0176 - val_accuracy: 0.9976
Epoch 6/10
66/66 [==============================] - 6s 87ms/step - loss: 0.0231 - accuracy: 0.9951 - val_loss: 0.0147 - val_accuracy: 0.9977
Epoch 7/10
66/66 [==============================] - 6s 89ms/step - loss: 0.0185 - accuracy: 0.9954 - val_loss: 0.0128 - val_accuracy: 0.9978
Epoch 8/10


In [10]:
# Step 5: Model Evaluation
_, accuracy = model.evaluate(X_test, y_test, batch_size=batch_size)
print('Test Accuracy:', accuracy)


14/14 [==============================] - 1s 30ms/step - loss: 0.0323 - accuracy: 0.9941
Test Accuracy: 0.9940977692604065


In [11]:
import numpy as np
from sklearn.metrics import f1_score

# Perform NER prediction
predictions = model.predict(X_test)
predicted_labels = np.argmax(predictions, axis=-1)

# Flatten the predicted_labels array
predicted_labels_flat = predicted_labels.flatten()

# Reshape X_test to match the dimensions of predicted_labels_flat
X_test_reshaped = np.reshape(X_test, (X_test.shape[0] * X_test.shape[1],))

# Ensure the lengths of X_test_reshaped and predicted_labels_flat are the same
min_len = min(len(X_test_reshaped), len(predicted_labels_flat))
X_test_reshaped = X_test_reshaped[:min_len]
predicted_labels_flat = predicted_labels_flat[:min_len]

# Convert y_test to one-hot encoding
y_true = np.argmax(y_test[:min_len], axis=-1).flatten()

# Compute the F1 score for the NER task with minimum span 0 and maximum span 4
f1_score_span_0_4 = f1_score(y_true, predicted_labels_flat, average='micro')
print('F1 Score (Span 0-4):', f1_score_span_0_4)


14/14 [==============================] - 2s 31ms/step
F1 Score (Span 0-4): 0.9940977507187553
